# 108 Python intermediate - dekoratory
_Kamil Bartocha_ | _wersja 2.0_

## Rozkład jazdy

1. 🔹 **Podstawy dekoratorów** - funkcja jako argument, wzorzec wrapper
2. 🔹 **Dekoratory z parametrami** - fabryka dekoratorów, `functools.wraps`
3. 🔹 **Wbudowane dekoratory** - `@staticmethod`, `@classmethod`, `@property`
4. 🔹 **@dataclass** - automatyczne generowanie klas danych

✏️ Każda sekcja zawiera ćwiczenia.

## 1. 🔹 Podstawy dekoratorów

Dekorator (decorator) to funkcja, która przyjmuje inną funkcję jako
argument i zwraca nową funkcję z rozszerzonym zachowaniem. Pozwala
dodawać funkcjonalność bez modyfikowania oryginalnego kodu.

Składnia `@decorator` to skrót - oba zapisy są równoważne:

```python
@my_decorator          # skrót
def greet(): ...

def greet(): ...       # to samo bez skrótu
greet = my_decorator(greet)
```

Schemat budowy dekoratora:

```
def decorator(func):       # 1. przyjmuje funkcję
    def wrapper(*args, **kwargs):
        # 2. kod przed
        result = func(*args, **kwargs)
        # 3. kod po
        return result
    return wrapper         # 4. zwraca nową funkcję
```

> 💡 **Tip:** `*args, **kwargs` w `wrapper` sprawia, że dekorator
> działa z dowolną funkcją - niezależnie od jej sygnaturę.

In [ ]:
import time

# Dekorator mierzacy czas wykonania
def timer(func):
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        elapsed = time.time() - start
        print(f"{func.__name__} took {elapsed:.4f}s")
        return result
    return wrapper


@timer
def slow_sum(n):
    return sum(range(n))


print(slow_sum(1_000_000))   # slow_sum took 0.xxs

In [ ]:
# Dekorator logujacy wywolania
def log_calls(func):
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__}({args}, {kwargs})")
        result = func(*args, **kwargs)
        print(f"-> returned {result}")
        return result
    return wrapper


@log_calls
def add(a, b):
    return a + b


add(3, 5)
add(a=10, b=20)

---

### ✏️ Ćwiczenia - Podstawy dekoratorów

**Ćw. 1.** Napisz dekorator `shout`, który otacza wynik funkcji
wykrzyknikami: `"hello"` → `"!!! hello !!!"`.

**Ćw. 2.** Napisz dekorator `print_args`, który przed wywołaniem
funkcji wypisuje typy wszystkich argumentów pozycyjnych.

**Ćw. 3. *(Trudniejsze)*** Napisz dekorator `count_calls`, który
zlicza ile razy dekorowana funkcja została wywołana. Przechowaj
licznik jako atrybut `wrapper.calls`.

In [ ]:
# Ćw. 1: shout
def shout(func):
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        ...
    return wrapper


@shout
def greet(name):
    return f"hello {name}"


print(greet("Alice"))   # !!! hello Alice !!!

In [ ]:
# Ćw. 2: print_args
def print_args(func):
    def wrapper(*args, **kwargs):
        for arg in args:
            ...
        return func(*args, **kwargs)
    return wrapper


@print_args
def echo(value):
    return value


echo("hello")   # arg type: str
echo(42)        # arg type: int

In [ ]:
# Ćw. 3: count_calls
def count_calls(func):
    def wrapper(*args, **kwargs):
        wrapper.calls += 1
        return func(*args, **kwargs)
    wrapper.calls = 0
    return wrapper


@count_calls
def say_hi():
    print("hi")


say_hi()
say_hi()
say_hi()
print(say_hi.calls)   # 3

## 2. 🔹 Dekoratory z parametrami

Gdy chcemy przekazać parametry do dekoratora, dodajemy jeszcze jeden
poziom zagnieżdżenia - zewnętrzna funkcja przyjmuje parametry i zwraca
właściwy dekorator:

```
def repeat(n):             # 1. przyjmuje parametr
    def decorator(func):   # 2. właściwy dekorator
        def wrapper(*args, **kwargs):
            for _ in range(n):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator       # 3. zwraca dekorator

@repeat(3)
def greet(): print("hello")
```

### `functools.wraps` - zachowanie metadanych

Dekorator bez `@wraps` nadpisuje `__name__` i `__doc__` oryginalnej
funkcji. `@wraps(func)` kopiuje te metadane do `wrapper`:

> 💡 **Tip:** Zawsze używaj `@wraps(func)` wewnątrz dekoratora.
> Ułatwia debugowanie i sprawia, że `help()` działa poprawnie.

In [ ]:
from functools import wraps

# Bez @wraps - metadane sa nadpisywane
def bad_decorator(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

# Z @wraps - metadane sa zachowane
def good_decorator(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper


@bad_decorator
def greet():
    """Say hello."""
    return "hello"

@good_decorator
def farewell():
    """Say goodbye."""
    return "goodbye"


print(greet.__name__)      # wrapper  (zepsute)
print(farewell.__name__)   # farewell (poprawne)

In [ ]:
from functools import wraps

# Dekorator z parametrem - kontrola uprawnien
def require_permission(level):
    def decorator(func):
        @wraps(func)
        def wrapper(user, *args, **kwargs):
            if user["level"] < level:
                raise PermissionError(
                    f"Required level {level}, got {user['level']}"
                )
            return func(user, *args, **kwargs)
        return wrapper
    return decorator


@require_permission(2)
def view_report(user):
    return "Secret report"


admin = {"name": "Alice", "level": 3}
guest = {"name": "Bob",   "level": 1}

print(view_report(admin))   # Secret report
try:
    view_report(guest)
except PermissionError as e:
    print(e)

---

### ✏️ Ćwiczenia - Dekoratory z parametrami

**Ćw. 4.** Napisz dekorator `repeat(n)`, który wywołuje dekorowaną
funkcję `n` razy i zwraca wynik ostatniego wywołania.

**Ćw. 5.** Napisz dekorator `validate_type(expected_type)`, który
sprawdza czy pierwszy argument funkcji jest oczekiwanego typu.
Jeśli nie - zgłasza `TypeError`.

**Ćw. 6. *(Trudniejsze)*** Napisz dekorator `retry(times)`,
który ponawia wywołanie funkcji do `times` razy w przypadku
wyjątku i dopiero po wyczerpaniu prób propaguje błąd dalej.

In [ ]:
from functools import wraps

# Ćw. 4: repeat(n)
def repeat(n):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            result = None
            for _ in range(n):
                ...
            return result
        return wrapper
    return decorator


@repeat(3)
def ping():
    print("ping")
    return "ok"


result = ping()    # ping  ping  ping
print(result)      # ok

In [ ]:
from functools import wraps

# Ćw. 5: validate_type
def validate_type(expected_type):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            if args and not isinstance(args[0], expected_type):
                raise TypeError(
                    f"Expected {expected_type.__name__}, "
                    f"got {type(args[0]).__name__}"
                )
            return ...
        return wrapper
    return decorator


@validate_type(str)
def shout(text):
    return text.upper()


print(shout("hello"))   # HELLO
print(shout(123))       # TypeError

In [ ]:
from functools import wraps

# Ćw. 6: retry(times)
def retry(times):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            # hint: petla for i in range(times)
            ...
        return wrapper
    return decorator


attempt = 0

@retry(3)
def flaky():
    global attempt
    attempt += 1
    if attempt < 3:
        raise RuntimeError(f"Fail #{attempt}")
    return "success"


print(flaky())      # success (after 2 failures)

## 3. 🔹 Wbudowane dekoratory

Python udostępnia kilka wbudowanych dekoratorów dla metod klasy.
Różnią się pierwszym argumentem i sposobem wywołania:

| Dekorator | Pierwszy arg | Dostęp do | Wywołanie |
|-----------|-------------|-----------|----------|
| (brak) - zwykła metoda | `self` | instancja + klasa | `obj.method()` |
| `@classmethod` | `cls` | klasa (nie instancja) | `Cls.method()` |
| `@staticmethod` | brak | brak | `Cls.method()` |
| `@property` | `self` | instancja | `obj.attr` (bez `()`) |

In [ ]:
# @staticmethod - metoda niezalezna od instancji i klasy
class MathUtils:
    @staticmethod
    def add(a, b):
        return a + b

    @staticmethod
    def is_even(n):
        return n % 2 == 0


print(MathUtils.add(3, 4))     # 7  (bez tworzenia instancji)
print(MathUtils.is_even(10))   # True

In [ ]:
# @classmethod - metoda fabryczna (alternatywny konstruktor)
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius

    @classmethod
    def from_fahrenheit(cls, fahrenheit):
        return cls((fahrenheit - 32) * 5 / 9)

    @classmethod
    def from_kelvin(cls, kelvin):
        return cls(kelvin - 273.15)

    def __str__(self):
        return f"{self.celsius:.1f}°C"


t1 = Temperature.from_fahrenheit(212)
t2 = Temperature.from_kelvin(373.15)
print(t1)   # 100.0°C
print(t2)   # 100.0°C

In [ ]:
# @property - getter i setter bez wywolania z nawiasami
class Circle:
    def __init__(self, radius):
        self._radius = radius

    @property
    def radius(self):
        return self._radius

    @radius.setter
    def radius(self, value):
        if value < 0:
            raise ValueError("Radius cannot be negative")
        self._radius = value

    @property
    def area(self):
        import math
        return math.pi * self._radius ** 2


c = Circle(5)
print(c.radius)       # 5  (wywolanie jak atrybut, nie metoda)
print(f"{c.area:.2f}")  # 78.54
c.radius = 10
print(c.radius)       # 10
c.radius = -1         # ValueError

---

### ✏️ Ćwiczenia - Wbudowane dekoratory

**Ćw. 7.** Napisz klasę `Converter` z metodami statycznymi:
- `km_to_miles(km)` - konwersja kilometrów na mile (1 km = 0.621371)
- `celsius_to_fahrenheit(c)` - konwersja temperatur

**Ćw. 8.** Napisz klasę `Date` z metodami klasowymi:
- `from_string(s)` - tworzy obiekt z napisu `"2026-03-24"`
- `today()` - tworzy obiekt z bieżącą datą (`datetime.date.today()`)

**Ćw. 9. *(Trudniejsze)*** Napisz klasę `Rectangle` z `@property`:
- `width` i `height` z walidacją `> 0`
- tylko do odczytu `area` i `perimeter`

In [ ]:
# Ćw. 7: Converter - @staticmethod
class Converter:
    @staticmethod
    def km_to_miles(km):
        ...

    @staticmethod
    def celsius_to_fahrenheit(c):
        ...


print(Converter.km_to_miles(100))         # 62.1371
print(Converter.celsius_to_fahrenheit(0)) # 32.0

In [ ]:
from datetime import date

# Ćw. 8: Date - @classmethod
class Date:
    def __init__(self, year, month, day):
        self.year  = year
        self.month = month
        self.day   = day

    @classmethod
    def from_string(cls, s):
        # hint: s.split('-') -> int(year), int(month), int(day)
        ...

    @classmethod
    def today(cls):
        ...

    def __str__(self):
        return f"{self.year}-{self.month:02d}-{self.day:02d}"


d = Date.from_string("2026-03-24")
print(d)            # 2026-03-24
print(Date.today()) # aktualna data

In [ ]:
# Ćw. 9: Rectangle z @property
class Rectangle:
    def __init__(self, width, height):
        self._width  = None
        self._height = None
        self.width   = width
        self.height  = height

    @property
    def width(self):
        return self._width

    @width.setter
    def width(self, value):
        if value <= 0:
            raise ValueError("Width must be positive")
        ...

    @property
    def height(self):
        ...

    @height.setter
    def height(self, value):
        ...

    @property
    def area(self):
        ...

    @property
    def perimeter(self):
        ...


r = Rectangle(4, 6)
print(r.area)       # 24
print(r.perimeter)  # 20
r.width = -1        # ValueError

## 4. 🔹 @dataclass

Dekorator `@dataclass` z modułu `dataclasses` automatycznie generuje
metody `__init__`, `__repr__` i `__eq__` na podstawie adnotacji typów.
Eliminuje powtarzalny kod w klasach przechowujących dane.

| Bez @dataclass | Z @dataclass |
|----------------|-------------|
| ręczny `__init__` | generowany automatycznie |
| ręczny `__repr__` | generowany automatycznie |
| ręczny `__eq__` | generowany automatycznie |
| mutowalny | mutowalny (lub `frozen=True`) |

```python
@dataclass
class Point:
    x: float
    y: float
    label: str = ""   # wartość domyślna
```

> 💡 **Tip:** `@dataclass(frozen=True)` tworzy niemutowalny obiekt
> (jak `namedtuple`). Można go używać jako klucz słownika lub element
> zbioru, bo jest hashable.

In [ ]:
from dataclasses import dataclass

# Porownanie: bez i z @dataclass

# Bez dekoratora - duzo boilerplate
class PointOld:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f"Point(x={self.x}, y={self.y})"

    def __eq__(self, other):
        return self.x == other.x and self.y == other.y


# Z dekoratorem - zwiezly zapis
@dataclass
class Point:
    x: float
    y: float


p1 = Point(1.0, 2.0)
p2 = Point(1.0, 2.0)
print(p1)          # Point(x=1.0, y=2.0)
print(p1 == p2)    # True
p1.x = 99          # mozna modyfikowac

In [ ]:
from dataclasses import dataclass, field

# frozen=True - obiekt niemutowalny
@dataclass(frozen=True)
class Color:
    r: int
    g: int
    b: int

    def to_hex(self):
        return f"#{self.r:02x}{self.g:02x}{self.b:02x}"


red = Color(255, 0, 0)
print(red)            # Color(r=255, g=0, b=0)
print(red.to_hex())   # #ff0000

try:
    red.r = 100       # FrozenInstanceError
except Exception as e:
    print(e)

# frozen=True -> hashable -> mozna uzyc jako klucz slownika
palette = {red: "primary red"}
print(palette[Color(255, 0, 0)])   # primary red

---

### ✏️ Ćwiczenia - @dataclass

**Ćw. 10.** Napisz klasę `Book` z polami `title`, `author`, `year`
używając `@dataclass`. Dodaj metodę `age()` zwracającą wiek książki
(2026 - year).

**Ćw. 11.** Napisz `@dataclass(frozen=True)` klasę `Coordinate`
z polami `lat` i `lon` (float). Sprawdź, że dwa obiekty z tymi
samymi wartościami są równe i że można użyć obiektu jako klucza
słownika.

**Ćw. 12. *(Trudniejsze)*** Napisz `@dataclass` klasę `Student`
z polami `name`, `grades` (lista, domyślnie pusta - użyj `field`).
Dodaj metodę `average()` zwracającą średnią ocen lub `0.0`
gdy lista jest pusta.

In [ ]:
from dataclasses import dataclass

# Ćw. 10: Book
@dataclass
class Book:
    title:  str
    author: str
    year:   int

    def age(self):
        ...


b = Book("1984", "Orwell", 1949)
print(b)          # Book(title='1984', author='Orwell', year=1949)
print(b.age())    # 77

In [ ]:
from dataclasses import dataclass

# Ćw. 11: Coordinate (frozen)
@dataclass(frozen=True)
class Coordinate:
    lat: float
    lon: float


warsaw  = Coordinate(52.2297, 21.0122)
warsaw2 = Coordinate(52.2297, 21.0122)
print(warsaw == warsaw2)   # True

locations = {warsaw: "Warsaw"}
print(locations[warsaw2])  # Warsaw

In [ ]:
from dataclasses import dataclass, field

# Ćw. 12: Student z lista ocen
@dataclass
class Student:
    name:   str
    grades: list = field(default_factory=list)

    def average(self):
        # hint: zwroc 0.0 gdy lista pusta
        ...


s = Student("Alice")
s.grades = [90, 85, 92]
print(s.average())           # 89.0
print(Student("Bob").average())  # 0.0